In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

In [ ]:
project_id = 'oceanic-citadel-454608-d2'
from google.cloud import bigquery
client = bigquery.Client(project=project_id)

In [ ]:
import pandas as pd
import numpy as np

# ── Date anchor ──
# AS_OF_DATE = pd.Timestamp.now().normalize() - pd.Timedelta(days=2)
AS_OF_DATE = pd.Timestamp('2026-06-08')

# ── Goal horizons & checkpoints ──
GOAL_HORIZONS = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
CHECKPOINTS   = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]

# ── Lookback window ──
LOOKBACK_COHORTS = 35

# ── Bucket label ──
ORGANIC_LABEL = 'Organic'

# ── Trim configs for organic share testing ──
# method: 'none' | 'winsor' | 'cohort_trim'
# persistent_trim only applies to cohort_trim (excluded users carry forward)
TRIM_CONFIGS = [
    {'method': 'none',        'pct': 0.0,  'label': 'notrim'},
    {'method': 'winsor',      'pct': 0.01, 'label': 'winsor_1pct'},
    {'method': 'winsor',      'pct': 0.02, 'label': 'winsor_2pct'},
    {'method': 'winsor',      'pct': 0.03, 'label': 'winsor_3pct'},
    {'method': 'winsor',      'pct': 0.04, 'label': 'winsor_4pct'},
    {'method': 'winsor',      'pct': 0.05, 'label': 'winsor_5pct'},
#    {'method': 'cohort_trim', 'pct': 0.05, 'label': 'trim_5pct'},
#    {'method': 'cohort_trim', 'pct': 0.10, 'label': 'trim_10pct'},
]

print(f'Config loaded. as_of_date = {AS_OF_DATE.date()}')

In [ ]:
from pandas_gbq import read_gbq

SQL_FLOOR_DATE = (
    AS_OF_DATE - pd.Timedelta(days=max(GOAL_HORIZONS) + LOOKBACK_COHORTS + 5)
).date()
print(f'SQL floor date: {SQL_FLOOR_DATE}  (AS_OF_DATE = {AS_OF_DATE.date()})')

# `analytics.lonestar_cost_per_user` is pre-filtered for test/marketing accounts.
# TikTok affids (4866 = TikTok, 7127 = TikTok Canada) are excluded entirely.
users_df = read_gbq(f"""
SELECT
  id,
  CASE
    WHEN affid IN (63, 4432, 4551, 4698, 5048, 5125, 7120, 7253, 7260, 8331, 8345) THEN 'Web'
    -- WHEN affid = 1 THEN 'App'   -- uncomment when LS App launches
    WHEN affid IN (64, 71)  THEN 'PPC'
    WHEN affid IN (0, 78)   THEN 'Organic'
    ELSE 'Affiliate'
  END AS population,
  DATE(MIN(cost_date)) AS cost_date
FROM `analytics.lonestar_cost_per_user`
WHERE cost_date >= DATE('{SQL_FLOOR_DATE}')
  AND affid NOT IN (4866, 7127)
  AND id > 0
GROUP BY 1, 2
""", project_id=project_id, use_bqstorage_api=True)

revenue_df = read_gbq(f"""
SELECT
  playerId AS playerid,
  DATE(date) AS date,
  SUM(amount) / 100.0 AS amount
FROM `lonestar.casino_astropay_dmn`
WHERE Status = 'APPROVED'
  AND date >= DATE('{SQL_FLOOR_DATE}')
GROUP BY 1, 2
""", project_id=project_id, use_bqstorage_api=True)

print(f'users_df:   {len(users_df):,} rows')
print(f'revenue_df: {len(revenue_df):,} rows')
print(users_df['population'].value_counts().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════
# ORGANIC SHARE — multi-config cohort progression with scope
# Supports winsor, cohort_trim (with persistent trim), or none.
# Percentiles always computed from depositors only.
#
# If users_df has 'scope' and 'bucket' columns, uses them directly
# (RP: app vs non_app). Otherwise defaults to scope='all' and
# derives bucket from population == ORGANIC_LABEL (LS).
# ══════════════════════════════════════════════════════════════

def organic_share_cohort_progression(
    users_df, revenue_df, *,
    as_of_date,
    goal_horizons=GOAL_HORIZONS,
    checkpoints=CHECKPOINTS,
    lookback_cohorts=LOOKBACK_COHORTS,
    positive_amount_only=True,
    trim_configs=TRIM_CONFIGS,
    persistent_trim=True,
):
    """
    For each goal horizon, fix a cohort of `lookback_cohorts` days ending at
    as_of_date − horizon. Measure organic vs acquired revenue split at every
    checkpoint ≤ horizon under each trim configuration.

    trim_configs: list of dicts with keys:
        'method': 'none' | 'winsor' | 'cohort_trim'
        'pct':    float (e.g. 0.05)
        'label':  str — column suffix in output

    persistent_trim: when True and method='cohort_trim', users excluded at
        an earlier checkpoint stay excluded for all later checkpoints within
        the same horizon. Winsor is unaffected (no user exclusion).

    Output columns per config (suffix = label):
      organic_sum_{label}, acquired_sum_{label}, total_sum_{label},
      organic_share_pct_{label}, users_org_{label}, users_acq_{label}
    """
    as_of_date = pd.to_datetime(as_of_date).normalize()

    has_scope = 'scope' in users_df.columns and 'bucket' in users_df.columns
    cols = ['id', 'cost_date']
    if has_scope:
        cols += ['scope', 'bucket']
    if 'population' in users_df.columns:
        cols.append('population')

    u = users_df[cols].copy()
    u['cost_date'] = pd.to_datetime(u['cost_date'], errors='coerce').dt.date
    u = u.loc[pd.notna(u['cost_date'])].copy()

    if not has_scope:
        u['scope'] = 'all'
        u['bucket'] = np.where(u['population'] == ORGANIC_LABEL, 'organic', 'acquired')

    u = u.drop_duplicates(subset=['id'])

    r = revenue_df[['playerid', 'date', 'amount']].copy()
    r['date'] = pd.to_datetime(r['date'], errors='coerce').dt.date
    r = r.loc[pd.notna(r['date'])].copy()
    if positive_amount_only:
        r = r[r['amount'] > 0]

    rr = r.merge(
        u.rename(columns={'id': '__uid__'}),
        left_on='playerid', right_on='__uid__', how='inner'
    )
    rr['dsi'] = (pd.to_datetime(rr['date']) - pd.to_datetime(rr['cost_date'])).dt.days
    rr = rr.loc[(rr['dsi'] >= 0) & (rr['dsi'] <= (max(checkpoints) - 1))].copy()

    daily_user = (
        rr.groupby(['scope', 'bucket', 'cost_date', '__uid__', 'dsi'], observed=True)['amount']
          .sum().reset_index()
          .sort_values(['scope', 'bucket', 'cost_date', '__uid__', 'dsi'])
    )
    daily_user['cum_amount'] = (
        daily_user.groupby(['scope', 'bucket', 'cost_date', '__uid__'], observed=True)['amount']
                  .cumsum()
    )

    rows = []
    scopes = sorted(u['scope'].unique())

    for horizon in goal_horizons:
        cohort_end   = (as_of_date - pd.Timedelta(days=horizon)).date()
        cohort_start = (as_of_date - pd.Timedelta(days=horizon + (lookback_cohorts - 1))).date()

        elig = u.loc[
            (u['cost_date'] >= cohort_start) &
            (u['cost_date'] <= cohort_end)
        ].copy()

        if elig.empty:
            print(f'[horizon={horizon}] No eligible users — skipping.')
            continue

        eligible_cps = sorted(c for c in checkpoints if c <= horizon)

        for scope in scopes:
            scope_elig = elig.loc[elig['scope'] == scope]
            if scope_elig.empty:
                continue

            scope_du_h = daily_user.merge(
                scope_elig[['scope', 'bucket', 'id', 'cost_date']].rename(columns={'id': '__uid__'}),
                on=['scope', 'bucket', '__uid__', 'cost_date'], how='inner'
            )

            excluded = {
                cfg['label']: set()
                for cfg in trim_configs
                if cfg['method'] == 'cohort_trim' and persistent_trim
            }

            for cp in eligible_cps:
                du_cp = scope_du_h.loc[scope_du_h['dsi'] <= (cp - 1)]

                row = dict(
                    scope          = scope,
                    goal_horizon   = horizon,
                    cohort_start   = cohort_start,
                    cohort_end     = cohort_end,
                    checkpoint_day = cp,
                )

                if du_cp.empty:
                    for cfg in trim_configs:
                        lbl = cfg['label']
                        row.update({
                            f'organic_sum_{lbl}'       : 0.0,
                            f'acquired_sum_{lbl}'      : 0.0,
                            f'total_sum_{lbl}'         : 0.0,
                            f'organic_share_pct_{lbl}' : np.nan,
                            f'users_org_{lbl}'         : 0,
                            f'users_acq_{lbl}'         : 0,
                        })
                    rows.append(row)
                    continue

                cum_cp = (
                    du_cp.groupby(['bucket', 'cost_date', '__uid__'], observed=True)['cum_amount']
                         .max().reset_index(name='cum_cp')
                )
                cum_cp = (
                    scope_elig.rename(columns={'id': '__uid__'})
                              .merge(cum_cp, on=['bucket', 'cost_date', '__uid__'], how='left')
                )
                cum_cp['cum_cp'] = cum_cp['cum_cp'].fillna(0.0)

                print_line = f'  [{scope}] D{cp:>3}:'

                for cfg in trim_configs:
                    method, pct, lbl = cfg['method'], cfg['pct'], cfg['label']

                    if method == 'none' or pct == 0.0:
                        kept = cum_cp.copy()

                    elif method == 'cohort_trim':
                        working = cum_cp.copy()
                        if persistent_trim and excluded.get(lbl):
                            working = working.loc[~working['__uid__'].isin(excluded[lbl])]

                        depositors = working.loc[working['cum_cp'] > 0]
                        if not depositors.empty:
                            thresh_map = (
                                depositors.groupby('bucket', observed=True)['cum_cp']
                                          .quantile(1.0 - pct)
                                          .to_dict()
                            )
                            thresh_series = working['bucket'].map(thresh_map).fillna(np.inf)
                            above_mask = (working['cum_cp'] > 0) & (working['cum_cp'] > thresh_series)
                            if persistent_trim:
                                excluded[lbl] |= set(working.loc[above_mask, '__uid__'].unique())
                            kept = working.loc[~above_mask]
                        else:
                            kept = working

                    elif method == 'winsor':
                        depositors = cum_cp.loc[cum_cp['cum_cp'] > 0]
                        if not depositors.empty:
                            cap_map = (
                                depositors.groupby('bucket', observed=True)['cum_cp']
                                          .quantile(1.0 - pct)
                                          .to_dict()
                            )
                            kept = cum_cp.copy()
                            caps = kept['bucket'].map(cap_map).fillna(np.inf)
                            kept['cum_cp'] = np.minimum(kept['cum_cp'], caps)
                        else:
                            kept = cum_cp.copy()

                    else:
                        raise ValueError(f"Unknown trim method: {method}")

                    sums = kept.groupby('bucket', observed=True)['cum_cp'].sum().to_dict()
                    cnts = kept.groupby('bucket', observed=True)['__uid__'].nunique().to_dict()

                    org_sum = float(sums.get('organic',  0.0))
                    acq_sum = float(sums.get('acquired', 0.0))
                    total   = org_sum + acq_sum
                    share   = (org_sum / total) if total > 0 else np.nan

                    row.update({
                        f'organic_sum_{lbl}'       : org_sum,
                        f'acquired_sum_{lbl}'      : acq_sum,
                        f'total_sum_{lbl}'         : total,
                        f'organic_share_pct_{lbl}' : share,
                        f'users_org_{lbl}'         : int(cnts.get('organic',  0)),
                        f'users_acq_{lbl}'         : int(cnts.get('acquired', 0)),
                    })

                    print_line += (f'  [{lbl}] org={org_sum:>10,.0f} '
                                   f'acq={acq_sum:>10,.0f} share={share:.1%}')

                print(print_line)
                rows.append(row)

    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values(['scope', 'goal_horizon', 'checkpoint_day']).reset_index(drop=True)
    id_cols = ['scope', 'goal_horizon', 'cohort_start', 'cohort_end', 'checkpoint_day']
    metric_cols = []
    for cfg in trim_configs:
        lbl = cfg['label']
        metric_cols += [
            f'organic_sum_{lbl}', f'acquired_sum_{lbl}', f'total_sum_{lbl}',
            f'organic_share_pct_{lbl}', f'users_org_{lbl}', f'users_acq_{lbl}',
        ]
    return df[id_cols + metric_cols]


print('Organic share function defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# RUN — Organic share cohort progression (multi-config)
# ══════════════════════════════════════════════════════════════

print('Computing organic share for all goal horizons...')
progression = organic_share_cohort_progression(
    users_df, revenue_df,
    as_of_date=AS_OF_DATE,
    goal_horizons=GOAL_HORIZONS,
    checkpoints=CHECKPOINTS,
    lookback_cohorts=LOOKBACK_COHORTS,
    positive_amount_only=True,
    trim_configs=TRIM_CONFIGS,
    persistent_trim=True,
)

print('\n' + '=' * 80)
print('ORGANIC SHARE COHORT PROGRESSION  (multi-config)')
print('=' * 80)
share_cols = [f'organic_share_pct_{cfg["label"]}' for cfg in TRIM_CONFIGS]
preview_cols = ['scope', 'goal_horizon', 'checkpoint_day'] + share_cols
for scope in progression['scope'].unique():
    print(f'\n── scope: {scope} ──')
    print(progression.loc[progression['scope'] == scope, preview_cols].to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# EXPORT
# ══════════════════════════════════════════════════════════════

out_name = f'lonestar_organic_share AS_OF_DATE_{AS_OF_DATE.strftime("%Y_%m_%d")}.csv'
progression.to_csv(out_name, index=False)

print(f'Saved: {out_name}')

from google.colab import files
files.download(out_name)